# Warm-ups: weeks 1 and 2

One small exercise for each study day, from **Mon 16 Nov to Fri 27 Nov**. Each takes about 30 minutes.

**How to use it**
1. Run the **setup cell** below once. If anything acts strangely later, use *Restart & Run All*.
2. Go to today's day. Read the short explanation.
3. Write your answer in the exercise cell (replace the `# your code here` / `raise NotImplementedError` lines) and run it.
4. Run the **check cell** under it. It says "passed" or tells you what is wrong.
5. Stuck for 10 minutes? Open the hint. Still stuck after 20? Paste your code to Claude and ask for a hint.

Do not look for the answer online first. A wrong attempt teaches you more than a copied answer.

The SQL days use a small practice table of **made-up** hourly electricity load (not real data).

In [ ]:
import math, sqlite3, contextlib
import pandas as pd

# ---- You don't need to read this cell, just run it. ----
# It builds a small practice database called load_hourly:
#   date (text), hour (0-23), load_mw (number), holiday (1 = holiday, 0 = not)
# 14 days x 24 hours = 336 rows of MADE-UP data, starting Mon 5 Jan 2026.

def _make_rows():
    rows = []
    start = pd.Timestamp("2026-01-05")
    for d in range(14):
        day = start + pd.Timedelta(days=d)
        date = day.strftime("%Y-%m-%d")
        holiday = 1 if date == "2026-01-07" else 0
        factor = 0.88 if day.dayofweek == 6 else 0.92 if day.dayofweek == 5 else 1.0
        if holiday:
            factor = 0.9
        for hour in range(24):
            base = 1400 + 350 * math.exp(-((hour - 19) / 3.0) ** 2) + 150 * math.exp(-((hour - 11) / 3.5) ** 2)
            wiggle = ((d * 7 + hour * 13) % 11) - 5
            rows.append((date, hour, round(base * factor + 4 * d + wiggle + 0.37 * d + 0.011 * hour, 1), holiday))
    return rows

_con = sqlite3.connect(":memory:")
_con.execute("CREATE TABLE load_hourly (date TEXT, hour INTEGER, load_mw REAL, holiday INTEGER)")
_con.executemany("INSERT INTO load_hourly VALUES (?, ?, ?, ?)", _make_rows())

def q(sql):
    """Run a SQL query and return the result as a table."""
    return pd.read_sql_query(sql, _con)

def show(df):
    try:
        from IPython.display import display
        display(df)
    except Exception:
        print(df)

load_df = q("SELECT * FROM load_hourly")   # the same table, as a pandas DataFrame (used by the checks)

def same_table(result, expected, sort_by=None):
    """Compare your query result with the expected table."""
    assert list(result.columns) == list(expected.columns), \
        f"Column names should be {list(expected.columns)} but you have {list(result.columns)}"
    assert len(result) == len(expected), f"Expected {len(expected)} rows but you got {len(result)}"
    if sort_by:
        result = result.sort_values(sort_by).reset_index(drop=True)
        expected = expected.sort_values(sort_by).reset_index(drop=True)
    for c in expected.columns:
        for i, (a, b) in enumerate(zip(result[c].tolist(), expected[c].tolist())):
            if isinstance(b, float):
                assert math.isclose(a, b, rel_tol=1e-6), f"Row {i}, column {c}: expected {b} but got {a}"
            else:
                assert a == b, f"Row {i}, column {c}: expected {b} but got {a}"

@contextlib.contextmanager
def raises(exc):
    try:
        yield
    except exc:
        return
    raise AssertionError(f"expected {exc.__name__} but nothing was raised")

print("Setup done. Table load_hourly has", len(load_df), "rows.")

## Day 1 · Mon 16 Nov · Python

**Goal:** practise loops and `if` with plain lists of numbers.

Write two small functions:
- `mean(xs)` returns the average of a list. An empty list should raise `ValueError`.
- `largest(xs)` returns the biggest number **without using `max()`**. An empty list should raise `ValueError`.

In [ ]:
def mean(xs):
    """Average of a list. Example: mean([1, 2, 3]) -> 2.0. Empty list -> ValueError."""
    # your code here
    raise NotImplementedError


def largest(xs):
    """Biggest number, without max(). Example: largest([3, 9, 2]) -> 9. Empty list -> ValueError."""
    # your code here
    raise NotImplementedError

**Hint:** for `mean`, add up the numbers (`sum`) and divide by how many there are (`len`). For `largest`, start with the first number as your best so far. Then go through the rest of the list, and each time you see a bigger one, make it your new best. Return the best at the end.

In [ ]:
def check_day1():
    assert mean([1, 2, 3]) == 2.0
    assert mean([10]) == 10
    assert mean([1.5, 2.5]) == 2.0
    with raises(ValueError):
        mean([])
    assert largest([3, 9, 2]) == 9
    assert largest([-5, -2, -9]) == -2
    assert largest([4]) == 4
    with raises(ValueError):
        largest([])
    print("Day 1 passed")
check_day1()

## Day 2 · Tue 17 Nov · SQL

**Goal:** read data from a table with `SELECT`.

SQL is a language for asking questions to a table. Three words to know today:
- `SELECT` says **which columns** you want
- `FROM` says **which table**
- `LIMIT` says **how many rows** at most

Example: `SELECT hour, load_mw FROM load_hourly LIMIT 3`

The table is called `load_hourly` and has the columns `date`, `hour`, `load_mw`, `holiday`. Run the next cell first to look at it.

In [ ]:
show(q("SELECT * FROM load_hourly LIMIT 5"))

In [ ]:
# Task A: show the columns date, hour and load_mw for the first 10 rows.
query_2a = """
-- write your query here
"""

# Task B: list every date once (14 dates in total). Show only the date column.
query_2b = """
-- write your query here
"""

**Hint:** Task A is `SELECT` (three column names separated by commas) `FROM load_hourly LIMIT 10`. For task B, put the word `DISTINCT` right after `SELECT`. It removes repeated values, so each date appears once.

In [ ]:
def check_day2():
    a = q(query_2a)
    same_table(a, load_df[["date", "hour", "load_mw"]].head(10).reset_index(drop=True))
    b = q(query_2b)
    same_table(b, pd.DataFrame({"date": sorted(load_df["date"].unique())}), sort_by="date")
    print("Day 2 passed")
show(q(query_2a))
check_day2()

## Day 3 · Wed 18 Nov · Python

**Goal:** loops and counting.

Write two functions:
- `count_above(xs, threshold)` counts how many numbers are **strictly bigger** than the threshold.
- `peak_hour(loads)` takes a list of 24 hourly loads and returns the **position** (index) of the biggest one. If two are tied, return the first.

In [ ]:
def count_above(xs, threshold):
    """How many numbers are > threshold. Example: count_above([1, 5, 10, 15], 5) -> 2"""
    # your code here
    raise NotImplementedError


def peak_hour(loads):
    """Position of the biggest value. Example: peak_hour([10, 30, 20]) -> 1. Empty list -> ValueError."""
    # your code here
    raise NotImplementedError

**Hint:** for `count_above`, start a counter at 0, go through the list, and add 1 when `x > threshold`. For `peak_hour`, `enumerate(loads)` gives you the position and the value together. Keep track of the best value and its position.

In [ ]:
def check_day3():
    assert count_above([1, 5, 10, 15], 5) == 2
    assert count_above([], 5) == 0
    assert count_above([5, 5], 5) == 0
    assert peak_hour([10, 30, 20]) == 1
    assert peak_hour([5, 9, 9, 1]) == 1
    assert peak_hour([7]) == 0
    with raises(ValueError):
        peak_hour([])
    print("Day 3 passed")
check_day3()

## Day 4 · Thu 19 Nov · SQL

**Goal:** pick rows with `WHERE` and sort them with `ORDER BY`.

- `WHERE load_mw > 1750` keeps only the rows that match the condition
- `ORDER BY load_mw DESC` sorts from biggest to smallest (`ASC` is smallest to biggest, and the default)
- You can use `AND`, `OR`, `=`, `<`, `>`

Example: `SELECT date, hour FROM load_hourly WHERE hour = 19 LIMIT 3`

In [ ]:
# Task A: show date, hour and load_mw for every row where load_mw is bigger than 1750.
query_4a = """
-- write your query here
"""

# Task B: show date, hour and load_mw for the 5 rows with the HIGHEST load_mw, biggest first.
query_4b = """
-- write your query here
"""

**Hint:** the order of the words matters: `SELECT ... FROM ... WHERE ... ORDER BY ... LIMIT ...`. Task A needs `WHERE`, task B needs `ORDER BY` and `LIMIT`.

In [ ]:
def check_day4():
    a = q(query_4a)
    exp_a = load_df[load_df["load_mw"] > 1750][["date", "hour", "load_mw"]].reset_index(drop=True)
    same_table(a, exp_a, sort_by=["date", "hour"])
    b = q(query_4b)
    exp_b = load_df.sort_values("load_mw", ascending=False).head(5)[["date", "hour", "load_mw"]].reset_index(drop=True)
    same_table(b, exp_b)
    print("Day 4 passed")
show(q(query_4b))
check_day4()

## Day 5 · Fri 20 Nov · Algorithm

**Goal:** work with text.

Write `is_palindrome(text)`. A palindrome reads the same forwards and backwards, **ignoring capital letters, spaces and punctuation**.

Examples: `"A man, a plan, a canal: Panama"` is `True`, `"hello"` is `False`.

In [ ]:
def is_palindrome(text):
    """True if text reads the same backwards, ignoring case, spaces and punctuation."""
    # your code here
    raise NotImplementedError

**Hint:** first build a clean list of characters: keep only letters and digits (`c.isalnum()`) and make them lowercase (`c.lower()`). Then compare that list with the same list reversed (`chars[::-1]`).

In [ ]:
def check_day5():
    assert is_palindrome("A man, a plan, a canal: Panama") is True
    assert is_palindrome("hello") is False
    assert is_palindrome("") is True
    assert is_palindrome("Was it a car or a cat I saw?") is True
    assert is_palindrome("ab") is False
    print("Day 5 passed")
check_day5()

## Day 6 · Mon 23 Nov · Python

**Goal:** build a new list from an old one (you will use this idea all the time in data science).

Write two functions:
- `normalize(xs)` rescales a list so the smallest becomes 0.0 and the biggest becomes 1.0. Formula for each value: `(x - smallest) / (biggest - smallest)`. An empty list, or a list where all numbers are equal, should raise `ValueError`.
- `clip(xs, low, high)` returns a new list where values below `low` become `low` and values above `high` become `high`.

In [ ]:
def normalize(xs):
    """Rescale to 0..1. Example: normalize([10, 20, 30]) -> [0.0, 0.5, 1.0]. Empty or all-equal -> ValueError."""
    # your code here
    raise NotImplementedError


def clip(xs, low, high):
    """Limit each value to the range low..high. Example: clip([1, 5, 10], 2, 8) -> [2, 5, 8]"""
    # your code here
    raise NotImplementedError

**Hint:** a list comprehension builds the new list in one line: `[ ... for x in xs ]`. For `clip`, the middle value of `min(...)` and `max(...)` does the job: `max(low, min(x, high))`.

In [ ]:
def check_day6():
    assert normalize([10, 20, 30]) == [0.0, 0.5, 1.0]
    assert normalize([5, 15]) == [0.0, 1.0]
    with raises(ValueError):
        normalize([])
    with raises(ValueError):
        normalize([4, 4])
    assert clip([1, 5, 10], 2, 8) == [2, 5, 8]
    assert clip([], 0, 1) == []
    print("Day 6 passed")
check_day6()

## Day 7 · Tue 24 Nov · SQL

**Goal:** summarise many rows into one number with *aggregate functions*.

- `COUNT(*)` counts rows
- `AVG(col)` is the average, `MAX(col)` the biggest, `MIN(col)` the smallest, `SUM(col)` the total
- `AS name` gives the result column a name

Example: `SELECT MIN(load_mw) AS lowest FROM load_hourly`

You can combine them with `WHERE`, e.g. `SELECT COUNT(*) AS n FROM load_hourly WHERE hour = 19`

In [ ]:
# Task A: ONE row with three columns:
#   n_rows (how many rows in the table), avg_load (average of load_mw), peak (biggest load_mw)
query_7a = """
-- write your query here
"""

# Task B: ONE row with one column called n: how many rows are holiday rows (holiday = 1)?
query_7b = """
-- write your query here
"""

**Hint:** task A has three aggregates separated by commas, each with its own `AS` name. Task B is `COUNT(*) AS n` plus a `WHERE` on the `holiday` column (`holiday = 1`).

In [ ]:
def check_day7():
    a = q(query_7a)
    exp_a = pd.DataFrame({"n_rows": [len(load_df)], "avg_load": [float(load_df["load_mw"].mean())], "peak": [float(load_df["load_mw"].max())]})
    same_table(a, exp_a)
    b = q(query_7b)
    same_table(b, pd.DataFrame({"n": [int((load_df["holiday"] == 1).sum())]}))
    print("Day 7 passed")
show(q(query_7a))
check_day7()

## Day 8 · Wed 25 Nov · Python

**Goal:** turn text into data and back (real data often arrives as text lines).

Write two functions:
- `parse_line(line)` turns text like `"2026-01-05,14,1450.5"` into a dictionary: `{"date": "2026-01-05", "hour": 14, "load": 1450.5}`. Extra spaces around the parts must be ignored. If the line does not have exactly 3 parts, raise `ValueError`.
- `format_report(rec)` turns such a dictionary into text like `"2026-01-05 07:00 -> 1450.5 MW"` (hour with 2 digits, load with 1 decimal).

In [ ]:
def parse_line(line):
    """'2026-01-05,14,1450.5' -> {'date': '2026-01-05', 'hour': 14, 'load': 1450.5}. Wrong number of parts -> ValueError."""
    # your code here
    raise NotImplementedError


def format_report(rec):
    """{'date': '2026-01-05', 'hour': 7, 'load': 1450.5} -> '2026-01-05 07:00 -> 1450.5 MW'"""
    # your code here
    raise NotImplementedError

**Hint:** `line.split(",")` cuts the text into pieces. `.strip()` removes spaces. `int("14")` and `float("1450.5")` convert text to numbers. For the report use an f-string: `f"{rec['hour']:02d}"` gives 2 digits, and `f"{x:.1f}"` gives 1 decimal.

In [ ]:
def check_day8():
    assert parse_line("2026-01-05,14,1450.5") == {"date": "2026-01-05", "hour": 14, "load": 1450.5}
    assert parse_line(" 2026-01-06 , 9 , 1300 ") == {"date": "2026-01-06", "hour": 9, "load": 1300.0}
    with raises(ValueError):
        parse_line("2026-01-05,14")
    assert format_report({"date": "2026-01-05", "hour": 7, "load": 1450.5}) == "2026-01-05 07:00 -> 1450.5 MW"
    assert format_report({"date": "2026-01-06", "hour": 19, "load": 1800.0}) == "2026-01-06 19:00 -> 1800.0 MW"
    print("Day 8 passed")
check_day8()

## Day 9 · Thu 26 Nov · SQL

**Goal:** summarise *per group* with `GROUP BY` (the most useful SQL idea in data science).

`GROUP BY date` makes one result row for each date, and the aggregates (`AVG`, `COUNT`, ...) are calculated inside each group.

Example: `SELECT hour, MAX(load_mw) AS peak FROM load_hourly GROUP BY hour ORDER BY hour`

`HAVING` works like `WHERE`, but it filters the **groups** after they are calculated (for example `HAVING AVG(load_mw) > 1000`).

In [ ]:
# Task A: the average load for each date. Columns: date, avg_load. Sort by date.
query_9a = """
-- write your query here
"""

# Task B: the average load for holiday vs not holiday. Columns: holiday, avg_load. Sort by holiday.
query_9b = """
-- write your query here
"""

# Challenge: only the dates whose average load is above 1500. Columns: date, avg_load. Sort by date.
query_9c = """
-- write your query here
"""

**Hint:** in tasks A and B, select the grouping column and `AVG(load_mw) AS avg_load`, then `GROUP BY` that column and `ORDER BY` it. For the challenge, add a `HAVING AVG(load_mw) > 1500` line after `GROUP BY`.

In [ ]:
def check_day9():
    g = load_df.groupby("date", as_index=False)["load_mw"].mean().rename(columns={"load_mw": "avg_load"})
    same_table(q(query_9a), g.reset_index(drop=True))
    h = load_df.groupby("holiday", as_index=False)["load_mw"].mean().rename(columns={"load_mw": "avg_load"})
    same_table(q(query_9b), h.reset_index(drop=True))
    c = g[g["avg_load"] > 1500].reset_index(drop=True)
    same_table(q(query_9c), c)
    print("Day 9 passed (including the challenge)")
show(q(query_9a))
check_day9()

## Day 10 · Fri 27 Nov · Algorithm

**Goal:** a classic interview question, solved the smart way.

Write `two_sum(nums, target)`. Find the **two different positions** whose numbers add up to `target`, and return them as a list `[i, j]` with `i < j`. You can assume there is exactly one answer.

Example: `two_sum([2, 7, 11, 15], 9)` is `[0, 1]` because 2 + 7 = 9.

In [ ]:
def two_sum(nums, target):
    """Positions [i, j] (i < j) of two numbers that add up to target. Example: two_sum([2, 7, 11, 15], 9) -> [0, 1]"""
    # your code here
    raise NotImplementedError

**Hint:** the slow way checks every pair with two loops. The fast way uses a dictionary: go through the list once; for each number, the one you still need is `target - number`. If that is already in your dictionary, you are done. If not, store this number and its position in the dictionary.

In [ ]:
def check_day10():
    assert two_sum([2, 7, 11, 15], 9) == [0, 1]
    assert two_sum([3, 2, 4], 6) == [1, 2]
    assert two_sum([3, 3], 6) == [0, 1]
    assert two_sum([-1, -2, -3, -4, -5], -8) == [2, 4]
    print("Day 10 passed")
check_day10()